In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
city_df_geo = pd.read_csv("data/city_safety/city_safety_2023_geo.csv")
country_df = pd.read_csv("data/crime-rate-by-country-2024.csv")

In [3]:
city_df = city_df_geo.copy()

# Clean country_norm on both sides
city_df['country'] = city_df['country'].astype(str).str.strip().str.lower()
country_df['country'] = country_df['country'].astype(str).str.strip().str.lower()

city_df['country_norm'] = city_df['country']
country_df['country_norm'] = country_df['country']

# Map state/province codes to countries
state_to_country = {
    'md': 'united states', 'tn': 'united states', 'mi': 'united states',
    'nm': 'united states', 'mo': 'united states', 'la': 'united states',
    'ca': 'united states', 'wi': 'united states', 'il': 'united states',
    'pa': 'united states', 'oh': 'united states', 'tx': 'united states',
    'ga': 'united states', 'ak': 'united states', 'dc': 'united states',
    'in': 'united states', 'ky': 'united states', 'fl': 'united states',
    'mn': 'united states', 'nv': 'united states', 'or': 'united states',
    'az': 'united states', 'wa': 'united states', 'ny': 'united states',
    'hi': 'united states', 'nc': 'united states', 'co': 'united states',
    'ma': 'united states', 'id': 'united states', 'ut': 'united states',
    'ab': 'canada', 'bc': 'canada',
}

name_fix = {
    'hong kong (china)': 'hong kong',
}

city_df['country_norm'] = (
    city_df['country_norm']
    .replace(state_to_country)
    .replace(name_fix)
)

In [5]:
# Standardize column names from the raw ta_data table
country_df = country_df.rename(columns={
    "Country": "country",
    "crimeRateByCountry_crimeIndex": "crime_index",
    "CrimeRate_OverallCriminalityScoreGOCI": "overall_criminality_score",
    "CrimeRate_CriminalMarketsScore": "criminal_markets_score",
    "CrimeRate_CriminalActorsScore": "criminal_actors_score",
    "CrimeRate_ResilienceScore": "resilience_score",
    "CrimeRateSafetyIndex": "safety_index",
})

country_df.columns.tolist()

['country',
 'crime_index',
 'overall_criminality_score',
 'criminal_markets_score',
 'criminal_actors_score',
 'resilience_score',
 'safety_index',
 'country_norm']

In [6]:
country_feats = [
    'country_norm',
    'crime_index',
    'overall_criminality_score',
    'criminal_markets_score',
    'criminal_actors_score',
    'resilience_score',
    'safety_index',
]

city_master = city_df.merge(
    country_df[country_feats],
    on='country_norm',
    how='left',
    suffixes=('', '_country')
)

print(city_master['crime_index_country'].isna().mean())

0.11057692307692307


In [7]:
missing_countries = (
    city_master.loc[city_master['crime_index_country'].isna(), 'country_norm']
    .value_counts()
)
print(missing_countries)

canada                37
colombia               3
thailand               3
dominican republic     1
moldova                1
georgia                1
Name: country_norm, dtype: int64


In [8]:
problem_list = [
    'canada', 'colombia', 'thailand',
    'dominican republic', 'moldova', 'georgia'
]

city_problem = (
    city_master
    .loc[city_master['country_norm'].isin(problem_list)
         & city_master['crime_index_country'].isna(),
         ['city', 'country', 'country_norm']]
    .sort_values(['country_norm', 'city'])
)

city_problem.head(80)

,city,country,country_norm
117,Brampton,canada,canada
337,Burlington,canada,canada
283,Calgary,canada,canada
387,Coquitlam,canada,canada
203,Edmonton,canada,canada
296,Guelph,canada,canada
243,Halifax,canada,canada
111,Hamilton,canada,canada
123,Kamloops,canada,canada
69,Kelowna,canada,canada


In [9]:
country_problem = country_df[
    country_df['country_norm'].isin(problem_list)
][['country', 'country_norm', 'crime_index']]

country_problem

,country,country_norm,crime_index
19,thailand,thailand,NaN
27,colombia,colombia,NaN
37,canada,canada,NaN
84,dominican republic,dominican republic,NaN
130,georgia,georgia,NaN
137,moldova,moldova,NaN


In [10]:
print(city_master.columns.tolist())

['Rank', 'city', 'country', 'crime_index', 'safety_index', 'year', 'city_norm', 'country_norm', 'lat', 'lon', 'crime_index_country', 'overall_criminality_score', 'criminal_markets_score', 'criminal_actors_score', 'resilience_score', 'safety_index_country']


## merging two city datasets with lat/lon

In [3]:
cities_2023 = pd.read_csv("../data/city_safety/city_safety_2023_geo_clean.csv")
wci_2022_geo = pd.read_csv("../data/city_safety/world_crime_index_2022_geo_clean.csv")

print("2023 rows:", len(cities_2023))
print("2022_geo rows:", len(wci_2022_geo))


2023 rows: 416
2022_geo rows: 453


In [4]:
# Build normalized keys for matching

def norm(s):
    return s.astype(str).str.strip().str.lower()

cities_2023["city_norm"] = norm(cities_2023["city"])
# if you already have country_norm in 2023, reuse it; otherwise normalize:
if "country_norm" not in cities_2023.columns:
    cities_2023["country_norm"] = norm(cities_2023["country"])

wci_2022_geo["city_norm"] = norm(wci_2022_geo["city"])
if "country_norm" not in wci_2022_geo.columns:
    wci_2022_geo["country_norm"] = norm(wci_2022_geo["country"])

In [5]:
# Identify 2022-only cities (no duplicate city+country)

keys_2023 = set(
    cities_2023[["city_norm", "country_norm"]]
    .itertuples(index=False, name=None)
)

mask_2022_only = ~wci_2022_geo[["city_norm", "country_norm"]].apply(tuple, axis=1).isin(keys_2023)
wci_2022_only = wci_2022_geo[mask_2022_only].copy()

print("2022-only cities (after geo):", len(wci_2022_only))

2022-only cities (after geo): 93


In [6]:
# Align columns and concatenate

# Core fields
cols_common = [
    "city",
    "country",
    "country_norm",
    "lat",
    "lon",
    "crime_index",
    "safety_index",
]

cities_2023_sub = cities_2023[cols_common].copy()
wci_2022_sub = wci_2022_only[cols_common].copy()

cities_all = pd.concat([cities_2023_sub, wci_2022_sub], ignore_index=True)

print("Merged cities_all rows:", len(cities_all))
print(cities_all.head())

Merged cities_all rows: 509
           city           country      country_norm        lat         lon  \
0       Caracas         Venezuela         venezuela  10.506093  -66.914601   
1      Pretoria      South Africa      south africa -25.745928   28.187910   
2        Durban      South Africa      south africa -29.861825   31.009909   
3  Port Moresby  Papua New Guinea  papua new guinea  -9.474330  147.159950   
4  Johannesburg      South Africa      south africa -26.205000   28.049722   

   crime_index  safety_index  
0         83.6          16.4  
1         82.0          18.0  
2         81.0          19.0  
3         80.7          19.3  
4         80.7          19.3  


In [7]:
# Save merged city dataset

out_path = "../data/city_safety/city_safety_2022_2023_merged_geo_clean.csv"
cities_all.to_csv(out_path, index=False)
print("Saved merged dataset to:", out_path)

Saved merged dataset to: ../data/city_safety/city_safety_2022_2023_merged_geo_clean.csv


# merging country level datasets

In [12]:
import pandas as pd

# crime 2023
c2023 = pd.read_csv("../data/country_safety/country_crime_pop_2023_clean.csv")  
c2023["country_norm"] = c2023["country"].astype(str).str.strip().str.lower()

# crime/safety 2020
c2020 = pd.read_csv("../data/country_safety/country_crime_safety_2020_clean.csv")
c2020["country_norm"] = c2020["country"].astype(str).str.strip().str.lower()

# age structure
age_df = pd.read_csv("../data/country_other/country_age_clean.csv")
age_df["country_norm"] = age_df["country"].astype(str).str.strip().str.lower()

# population density
pop_df = pd.read_csv("../data/country_other/country_pop_density_clean.csv")
pop_df["country_norm"] = pop_df["country"].astype(str).str.strip().str.lower()

In [18]:
print(c2023.columns)
print(c2020.columns)
print(c2023.shape)
print(c2020.shape)
print(age_df.shape)
print(pop_df.shape)

Index(['rank', 'country', 'crime_index_2023', 'population_2023',
       'country_norm'],
      dtype='object')
Index(['country', 'crime_index_2020', 'safety_index_2020', 'country_norm'], dtype='object')
(133, 5)
(129, 4)
(191, 5)
(251, 10)


In [16]:
c2023_small = c2023[["country_norm", "crime_index_2023"]].rename(
    columns={"crime_index_2023": "crimeindex_2023",
             "population_2023": "population_2023",
    }
)


c2020_small = c2020[["country_norm", "crime_index_2020", "safety_index_2020"]].rename(
    columns={
        "crime_index_2020": "crimeindex_2020",
        "safety_index_2020": "safetyindex_2020",
    }
)

age_small = age_df[["country_norm", "age_0_14", "age_15_64", "age_65_plus"]]

pop_small = pop_df[["country_norm", "population", "density_per_km2"]]

In [17]:
country_features = (
    c2023_small
    .merge(c2020_small, on="country_norm", how="outer")
    .merge(age_small, on="country_norm", how="outer")
    .merge(pop_small, on="country_norm", how="outer")
)

print("country_features shape:", country_features.shape)
print(country_features.head())
print(country_features.isna().mean())

country_features shape: (257, 9)
       country_norm  crimeindex_2023  crimeindex_2020  safetyindex_2020  \
0         venezuela            83.76            84.49             15.51   
1  papua new guinea            80.79            81.93             18.07   
2      south africa            76.86            77.49             22.51   
3       afghanistan            76.31            76.23             23.77   
4          honduras            74.54            76.11             23.89   

   age_0_14  age_15_64  age_65_plus  population  density_per_km2  
0      27.6       65.8          7.0  32219521.0             35.0  
1      35.9       60.3          4.0   8935000.0             19.0  
2      29.0       65.7          5.0  58775022.0             48.0  
3      43.2       54.2          3.0  31575018.0             49.0  
4      31.6       63.7          5.0   9158345.0             81.0  
country_norm        0.000000
crimeindex_2023     0.482490
crimeindex_2020     0.498054
safetyindex_2020    0.49805

In [20]:
country_features.isna().mean()

country_norm        0.000000
crimeindex_2023     0.482490
crimeindex_2020     0.498054
safetyindex_2020    0.498054
age_0_14            0.256809
age_15_64           0.256809
age_65_plus         0.256809
population          0.023346
density_per_km2     0.023346
dtype: float64

In [21]:
country_features.head()

,country_norm,crimeindex_2023,crimeindex_2020,safetyindex_2020,age_0_14,age_15_64,age_65_plus,population,density_per_km2
0,venezuela,83.76,84.49,15.51,27.6,65.8,7.0,32219521.0,35.0
1,papua new guinea,80.79,81.93,18.07,35.9,60.3,4.0,8935000.0,19.0
2,south africa,76.86,77.49,22.51,29.0,65.7,5.0,58775022.0,48.0
3,afghanistan,76.31,76.23,23.77,43.2,54.2,3.0,31575018.0,49.0
4,honduras,74.54,76.11,23.89,31.6,63.7,5.0,9158345.0,81.0


In [33]:
# merge country_features into cities_all
cities_all = pd.read_csv("../data/city_safety/city_safety_2022_2023_merged_geo_clean.csv")
cities_all["country_norm"] = cities_all["country"].astype(str).str.strip().str.lower()

# map states
state_to_country = {
    # US states
    "ak": "united states",
    "al": "united states",
    "az": "united states",
    "ar": "united states",
    "ca": "united states",
    "co": "united states",
    "ct": "united states",
    "de": "united states",
    "fl": "united states",
    "ga": "united states",
    "hi": "united states",
    "ia": "united states",
    "id": "united states",
    "il": "united states",
    "in": "united states",
    "ks": "united states",
    "ky": "united states",
    "la": "united states",
    "ma": "united states",
    "md": "united states",
    "me": "united states",
    "mi": "united states",
    "mn": "united states",
    "mo": "united states",
    "ms": "united states",
    "mt": "united states",
    "nc": "united states",
    "nd": "united states",
    "ne": "united states",
    "nh": "united states",
    "nj": "united states",
    "nm": "united states",
    "nv": "united states",
    "ny": "united states",
    "oh": "united states",
    "ok": "united states",
    "or": "united states",
    "pa": "united states",
    "ri": "united states",
    "sc": "united states",
    "sd": "united states",
    "tn": "united states",
    "tx": "united states",
    "ut": "united states",
    "va": "united states",
    "vt": "united states",
    "wa": "united states",
    "wi": "united states",
    "wv": "united states",
    "wy": "united states",
    "dc": "united states",
    # Canada provinces
    "ab": "canada",
    "bc": "canada",
    "mb": "canada",
    "nb": "canada",
    "nl": "canada",
    "ns": "canada",
    "nt": "canada",
    "nu": "canada",
    "on": "canada",
    "pe": "canada",
    "qc": "canada",
    "sk": "canada",
    "yt": "canada",
}

cities_all["country_norm"] = cities_all["country_norm"].replace(state_to_country)

name_fixes_country = {
    "trinidad and tobago": "trinidad and tobago",   
    "bosnia and herzegovina": "bosnia and herzegovina",  
    "hong kong (china)": "hong kong",  
}

country_features["country_norm"] = country_features["country_norm"].replace(name_fixes_country)

cities_all["country_norm"] = cities_all["country_norm"].replace({
    "hong kong (china)": "hong kong",
})

country_features["country_norm"] = country_features["country_norm"].replace(name_fixes_country)
cities_with_country = cities_all.merge(
    country_features,
    on="country_norm",
    how="left",
)

print("cities_with_country shape:", cities_with_country.shape)
print(cities_with_country.isna().mean())

cities_with_country shape: (509, 15)
city                0.000000
country             0.000000
country_norm        0.000000
lat                 0.000000
lon                 0.000000
crime_index         0.000000
safety_index        0.000000
crimeindex_2023     0.005894
crimeindex_2020     0.003929
safetyindex_2020    0.003929
age_0_14            0.001965
age_15_64           0.001965
age_65_plus         0.001965
population          0.003929
density_per_km2     0.003929
dtype: float64


In [34]:
# inspect missing country features BEFORE filling
country_feat_cols = [
    "crimeindex_2023",
    "crimeindex_2020",
    "safetyindex_2020",
    "age_0_14",
    "age_15_64",
    "age_65_plus",
    "population",
    "density_per_km2",
]

print("missing fraction per country feature (city level):")
print(cities_with_country[country_feat_cols].isna().mean())

missing fraction per country feature (city level):
crimeindex_2023     0.005894
crimeindex_2020     0.003929
safetyindex_2020    0.003929
age_0_14            0.001965
age_15_64           0.001965
age_65_plus         0.001965
population          0.003929
density_per_km2     0.003929
dtype: float64


In [35]:
missing_rows = cities_with_country[cities_with_country["crimeindex_2023"].isna()][
    ["city", "country", "country_norm"]
].drop_duplicates()

print("\nexample cities missing crimeindex_2023:")
print(missing_rows.head(20))


example cities missing crimeindex_2023:
              city                 country            country_norm
13   Port of Spain     Trinidad And Tobago     trinidad and tobago
202       Sarajevo  Bosnia And Herzegovina  bosnia and herzegovina
299     Banja Luka  Bosnia And Herzegovina  bosnia and herzegovina


In [36]:
# city rows with those indices
print(cities_all.loc[[13, 202, 299], ["city", "country", "country_norm"]])

# check if these country_norm values exist in any of your country tables
for name in ["trinidad and tobago", "bosnia and herzegovina"]:
    print("\n===", name, "===")
    print("in country_features:",
          country_features[country_features["country_norm"] == name])

              city                 country            country_norm
13   Port of Spain     Trinidad And Tobago     trinidad and tobago
202       Sarajevo  Bosnia And Herzegovina  bosnia and herzegovina
299     Banja Luka  Bosnia And Herzegovina  bosnia and herzegovina

=== trinidad and tobago ===
in country_features:             country_norm  crimeindex_2023  crimeindex_2020  safetyindex_2020  \
133  trinidad and tobago              NaN            73.19             26.81   

     age_0_14  age_15_64  age_65_plus  population  density_per_km2  
133      20.7       69.3         10.0   1363985.0            265.0  

=== bosnia and herzegovina ===
in country_features:                country_norm  crimeindex_2023  crimeindex_2020  \
134  bosnia and herzegovina              NaN            43.03   

     safetyindex_2020  age_0_14  age_15_64  age_65_plus  population  \
134             56.97      14.1       69.3         17.0   3511372.0   

     density_per_km2  
134             69.0  


In [37]:
import numpy as np

# fill 2023 crime with 2020 crime where 2023 is NaN and 2020 is not
mask_2023_nan_2020_ok = (
    cities_with_country["crimeindex_2023"].isna()
    & cities_with_country["crimeindex_2020"].notna()
)

cities_with_country.loc[mask_2023_nan_2020_ok, "crimeindex_2023"] = (
    cities_with_country.loc[mask_2023_nan_2020_ok, "crimeindex_2020"]
)

print("rows where we copied 2020 -> 2023:",
      mask_2023_nan_2020_ok.sum())

rows where we copied 2020 -> 2023: 3


In [40]:
country_feat_cols = [
    "crimeindex_2023",
    "crimeindex_2020",
    "safetyindex_2020",
    "age_0_14",
    "age_15_64",
    "age_65_plus",
    "population",
    "density_per_km2",
]

# which rows have at least one missing country feature
row_mask = cities_with_country[country_feat_cols].isna().any(axis=1)
problem_rows = cities_with_country.loc[row_mask, ["city", "country", "country_norm"] + country_feat_cols]

print("num city rows with any missing country feature:", len(problem_rows))
problem_rows.head(30)

num city rows with any missing country feature: 4


,city,country,country_norm,crimeindex_2023,crimeindex_2020,safetyindex_2020,age_0_14,age_15_64,age_65_plus,population,density_per_km2
33,San Juan,Puerto Rico,puerto rico,62.84,65.63,34.37,NaN,NaN,NaN,NaN,NaN
105,Montevideo,Uruguay,uruguay,51.73,53.81,46.19,21.1,64.3,15.0,NaN,NaN
493,Tashkent,Uzbekistan,uzbekistan,33.42,NaN,NaN,28.0,67.5,5.0,32653900.0,73.0
502,Kigali,Rwanda,rwanda,24.89,NaN,NaN,40.1,56.9,3.0,12374397.0,470.0


In [41]:
# convenience handle
df = cities_with_country

# Uzbekistan: fill crimeindex_2020 from 2023 and set safetyindex_2020 = 73.4
mask_uzb = df["country_norm"] == "uzbekistan"
df.loc[mask_uzb & df["crimeindex_2020"].isna(), "crimeindex_2020"] = \
    df.loc[mask_uzb & df["crimeindex_2020"].isna(), "crimeindex_2023"]

df.loc[mask_uzb, "safetyindex_2020"] = 73.4

# Rwanda: same for crime, safetyindex_2020 = 73.7
mask_rwa = df["country_norm"] == "rwanda"
df.loc[mask_rwa & df["crimeindex_2020"].isna(), "crimeindex_2020"] = \
    df.loc[mask_rwa & df["crimeindex_2020"].isna(), "crimeindex_2023"]

df.loc[mask_rwa, "safetyindex_2020"] = 73.7

# check those rows again
print(df.loc[mask_uzb | mask_rwa, [
    "city", "country", "crimeindex_2023", "crimeindex_2020", "safetyindex_2020"
]])

         city     country  crimeindex_2023  crimeindex_2020  safetyindex_2020
493  Tashkent  Uzbekistan            33.42            33.42              73.4
502    Kigali      Rwanda            24.89            24.89              73.7


In [42]:
row_mask = df[country_feat_cols].isna().any(axis=1)
print("rows still missing something after manual fixes:")
print(df.loc[row_mask, ["city", "country", "country_norm"] + country_feat_cols])

rows still missing something after manual fixes:
           city      country country_norm  crimeindex_2023  crimeindex_2020  \
33     San Juan  Puerto Rico  puerto rico            62.84            65.63   
105  Montevideo      Uruguay      uruguay            51.73            53.81   

     safetyindex_2020  age_0_14  age_15_64  age_65_plus  population  \
33              34.37       NaN        NaN          NaN         NaN   
105             46.19      21.1       64.3         15.0         NaN   

     density_per_km2  
33               NaN  
105              NaN  


In [43]:
mask_pr = cities_with_country["country_norm"] == "puerto rico"

cities_with_country.loc[mask_pr, "population"] = 3203792
cities_with_country.loc[mask_pr, "age_0_14"] = 15.9
cities_with_country.loc[mask_pr, "age_15_64"] = 63.8
cities_with_country.loc[mask_pr, "age_65_plus"] = 20.3
cities_with_country.loc[mask_pr, "density_per_km2"] = 361.0

In [44]:
mask_uru = cities_with_country["country_norm"] == "uruguay"

cities_with_country.loc[mask_uru, "population"] = 3388080
cities_with_country.loc[mask_uru, "density_per_km2"] = 19.36

In [45]:
country_cols = [
    "crimeindex_2023",
    "crimeindex_2020",
    "safetyindex_2020",
    "age_0_14",
    "age_15_64",
    "age_65_plus",
    "population",
    "density_per_km2",
]

row_mask = cities_with_country[country_cols].isna().any(axis=1)
cities_with_country.loc[row_mask, ["city","country","country_norm"] + country_cols]

,city,country,country_norm,crimeindex_2023,crimeindex_2020,safetyindex_2020,age_0_14,age_15_64,age_65_plus,population,density_per_km2


In [46]:
# Confirm no missing values anywhere
total_nans = cities_with_country.isna().sum().sum()
print("total NaNs in cities_with_country:", total_nans)

print("\nNaNs per column:")
print(cities_with_country.isna().sum())

# If total_nans is 0, inspect the first few rows
print("\nHead of final dataframe:")
cities_with_country.head()

total NaNs in cities_with_country: 0

NaNs per column (should all be 0):
city                0
country             0
country_norm        0
lat                 0
lon                 0
crime_index         0
safety_index        0
crimeindex_2023     0
crimeindex_2020     0
safetyindex_2020    0
age_0_14            0
age_15_64           0
age_65_plus         0
population          0
density_per_km2     0
dtype: int64

Head of final dataframe:


,city,country,country_norm,lat,lon,crime_index,safety_index,crimeindex_2023,crimeindex_2020,safetyindex_2020,age_0_14,age_15_64,age_65_plus,population,density_per_km2
0,Caracas,Venezuela,venezuela,10.506093,-66.914601,83.6,16.4,83.76,84.49,15.51,27.6,65.8,7.0,32219521.0,35.0
1,Pretoria,South Africa,south africa,-25.745928,28.187910,82.0,18.0,76.86,77.49,22.51,29.0,65.7,5.0,58775022.0,48.0
2,Durban,South Africa,south africa,-29.861825,31.009909,81.0,19.0,76.86,77.49,22.51,29.0,65.7,5.0,58775022.0,48.0
3,Port Moresby,Papua New Guinea,papua new guinea,-9.474330,147.159950,80.7,19.3,80.79,81.93,18.07,35.9,60.3,4.0,8935000.0,19.0
4,Johannesburg,South Africa,south africa,-26.205000,28.049722,80.7,19.3,76.86,77.49,22.51,29.0,65.7,5.0,58775022.0,48.0


In [49]:
# save cleaned dataframe
cities_with_country.to_csv(
    "../data/compiled_model_ready/MR_cities_with_country_v1.csv",
    index=False
)
print("  (:    ")

  (:    


# now merging 
### city_safety_2022_2023_merged_geo_clean 
## with crime-rate-by-country-2024-clean.csv

In [9]:
# Load merged city data (2022+2023)

cities_all = pd.read_csv("../data/city_safety/city_safety_2022_2023_merged_geo_clean.csv")
print("cities_all shape:", cities_all.shape)
print(cities_all.head())

# Ensure  normalized country key
if "country_norm" not in cities_all.columns:
    cities_all["country_norm"] = cities_all["country"].str.strip().str.lower()

cities_all shape: (509, 7)
           city           country      country_norm        lat         lon  \
0       Caracas         Venezuela         venezuela  10.506093  -66.914601   
1      Pretoria      South Africa      south africa -25.745928   28.187910   
2        Durban      South Africa      south africa -29.861825   31.009909   
3  Port Moresby  Papua New Guinea  papua new guinea  -9.474330  147.159950   
4  Johannesburg      South Africa      south africa -26.205000   28.049722   

   crime_index  safety_index  
0         83.6          16.4  
1         82.0          18.0  
2         81.0          19.0  
3         80.7          19.3  
4         80.7          19.3  


# merging v5 features with master dataset (509 cites)

In [4]:
import pandas as pd

macro_path = "../data/global_data/country_macro_v5.csv"
macro_v5 = pd.read_csv(macro_path)

macro_v5["country_norm"] = macro_v5["country_norm"].str.strip().str.lower()

# Check for duplicates in macro_v5
dupes = macro_v5["country_norm"][macro_v5["country_norm"].duplicated(keep=False)]
print("Duplicate country_norm values in macro_v5:")
print(dupes.value_counts())

# Inspect a few duplicate groups
for cn in dupes.unique():
    print(f"\n=== {cn} ===")
    print(macro_v5[macro_v5["country_norm"] == cn][
        ["name", "iso2", "region"]
    ].to_string(index=False))

# Collapse duplicates by keeping the first (you can also groupby/mean if needed)
macro_v5_dedup = (
    macro_v5
    .sort_values(["country_norm"])  # stable order
    .drop_duplicates(subset=["country_norm"], keep="first")
    .reset_index(drop=True)
)

print("\nmacro_v5_dedup shape:", macro_v5_dedup.shape)

Duplicate country_norm values in macro_v5:
china                  3
afghanistan            2
albania                2
algeria                2
andorra                2
angola                 2
antigua and barbuda    2
argentina              2
armenia                2
australia              2
austria                2
france                 2
turkey                 2
sweden                 2
united states          2
Name: country_norm, dtype: int64

=== afghanistan ===
       name iso2        region
Afghanistan   AF Southern Asia
Afghanistan   AF Southern Asia

=== albania ===
   name iso2          region
Albania   AL Southern Europe
Albania   AL Southern Europe

=== algeria ===
   name iso2          region
Algeria   DZ Northern Africa
Algeria   DZ Northern Africa

=== andorra ===
   name iso2          region
Andorra   AD Southern Europe
Andorra   AD Southern Europe

=== angola ===
  name iso2        region
Angola   AO Middle Africa
Angola   AO Middle Africa

=== antigua and barbuda ===


In [5]:
# Load v4 base and deduped macro_v5
base_path_v4 = "../data/compiled_model_ready/MR_cities_with_country_worldpop_knn_v1.csv"
macro_path = "../data/global_data/country_macro_v5.csv"

base_v4 = pd.read_csv(base_path_v4)
macro_v5 = pd.read_csv(macro_path)

base_v4["country_norm"] = base_v4["country"].str.strip().str.lower()
macro_v5["country_norm"] = macro_v5["country_norm"].str.strip().str.lower()

# Deduplicate macro_v5 on country_norm, keeping the first row for each country
macro_v5_dedup = (
    macro_v5
    .sort_values(["country_norm"])
    .drop_duplicates(subset=["country_norm"], keep="first")
    .reset_index(drop=True)
)

print("macro_v5_dedup shape:", macro_v5_dedup.shape)

# Macro columns for v5
macro_cols_v5 = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "homicide_rate",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "tourists",
]

# Merge macro features into v4 base
cities_v5 = base_v4.merge(
    macro_v5_dedup[["country_norm"] + macro_cols_v5],
    on="country_norm",
    how="left",
    validate="many_to_one",
)

print("cities_v5 shape (should be 509 rows):", cities_v5.shape)
print(cities_v5[["city", "country"] + macro_cols_v5].head())

print("\nNaNs in v5 macro columns (cities_v5):")
print(cities_v5[macro_cols_v5].isna().sum())

macro_v5_dedup shape: (192, 13)
cities_v5 shape (should be 509 rows): (509, 74)
           city           country       gdp  gdp_per_capita  unemployment  \
0       Caracas         Venezuela       NaN             NaN           NaN   
1      Pretoria      South Africa  368094.0          6369.2          28.5   
2        Durban      South Africa  368094.0          6369.2          28.5   
3  Port Moresby  Papua New Guinea   23077.0          2681.4           2.5   
4  Johannesburg      South Africa  368094.0          6369.2          28.5   

   homicide_rate  life_expectancy_male  life_expectancy_female  \
0            NaN                   NaN                     NaN   
1           36.4                  60.2                    67.1   
2           36.4                  60.2                    67.1   
3            9.8                  62.9                    65.4   
4           36.4                  60.2                    67.1   

   infant_mortality  urban_population_growth  tourists  
0  

In [6]:
# Which master countries have no matching country_norm in macro_v5_dedup?
missing_macro_countries = (
    base_v4[~base_v4["country_norm"].isin(macro_v5_dedup["country_norm"])]
    [["country", "country_norm"]]
    .drop_duplicates()
    .sort_values("country")
)

print("Countries in v4 with NO macro row (count =", len(missing_macro_countries), "):")
print(missing_macro_countries.to_string(index=False))

# Verify NaNs in cities_v5 line up with those countries
na_mask = cities_v5["gdp"].isna()
na_countries = (
    cities_v5[na_mask][["country", "country_norm"]]
    .drop_duplicates()
    .sort_values("country")
)

print("\nCountries in cities_v5 with NaN macro (via gdp) (count =", len(na_countries), "):")
print(na_countries.to_string(index=False))

Countries in v4 with NO macro row (count = 45 ):
          country      country_norm
               AB                ab
               AK                ak
               AZ                az
               BC                bc
               CA                ca
               CO                co
               DC                dc
               FL                fl
               GA                ga
               HI                hi
Hong Kong (China) hong kong (china)
               ID                id
               IL                il
               IN                in
             Iran              iran
               KY                ky
               LA                la
               MA                ma
               MD                md
               MI                mi
               MN                mn
               MO                mo
          Moldova           moldova
               NC                nc
               NM                nm
               

In [7]:
base_path_v4 = "../data/compiled_model_ready/MR_cities_with_country_worldpop_knn_v1.csv"
macro_path = "../data/global_data/country_macro_v5.csv"

base_v4 = pd.read_csv(base_path_v4)
macro_v5 = pd.read_csv(macro_path)

# Normalize macro table key
macro_v5["country_norm"] = macro_v5["country_norm"].str.strip().str.lower()

# Map v4 'country' to the names used in macro_v5/world_1
country_name_map = {
    "Russia": "Russian Federation",
    "Venezuela": "Venezuela, Bolivarian Republic Of",
    "South Korea": "Korea, Republic Of",
    "North Macedonia": "Macedonia, The Former Yugoslav Republic Of",
    "Moldova": "Moldova, Republic Of",
    "Iran": "Iran, Islamic Republic Of",
    "Syria": "Syrian Arab Republic",
    "Tanzania": "Tanzania, United Republic Of",
    "Vietnam": "Viet Nam",
    "United States": "United States",
    "United Kingdom": "United Kingdom",
    "Hong Kong (China)": "Hong Kong (China)",  
    "Taiwan": "Taiwan",                        
    "Puerto Rico": "Puerto Rico",              
    "Norway": "Norway",                        }

# Apply mapping only for merge; keep original 'country' unchanged
base_v4["country_for_macro"] = base_v4["country"].replace(country_name_map)
base_v4["country_for_macro_norm"] = base_v4["country_for_macro"].str.strip().str.lower()

In [8]:
# Deduplicate macro table on country_norm
macro_v5_dedup = (
    macro_v5
    .sort_values(["country_norm"])
    .drop_duplicates(subset=["country_norm"], keep="first")
    .reset_index(drop=True)
)

macro_cols_v5 = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "homicide_rate",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "tourists",
]

# Merge: many cities -> one macro row per country_for_macro_norm
cities_v5 = base_v4.merge(
    macro_v5_dedup[["country_norm"] + macro_cols_v5],
    left_on="country_for_macro_norm",
    right_on="country_norm",
    how="left",
    validate="many_to_one",
)

print("cities_v5 shape (should be 509 rows):", cities_v5.shape)
print(cities_v5[["city", "country"] + macro_cols_v5].head())

print("\nNaNs in v5 macro columns (cities_v5):")
print(cities_v5[macro_cols_v5].isna().sum())

cities_v5 shape (should be 509 rows): (509, 77)
           city           country       gdp  gdp_per_capita  unemployment  \
0       Caracas         Venezuela  208338.0          7212.2           9.4   
1      Pretoria      South Africa  368094.0          6369.2          28.5   
2        Durban      South Africa  368094.0          6369.2          28.5   
3  Port Moresby  Papua New Guinea   23077.0          2681.4           2.5   
4  Johannesburg      South Africa  368094.0          6369.2          28.5   

   homicide_rate  life_expectancy_male  life_expectancy_female  \
0           36.7                  68.4                    76.1   
1           36.4                  60.2                    67.1   
2           36.4                  60.2                    67.1   
3            9.8                  62.9                    65.4   
4           36.4                  60.2                    67.1   

   infant_mortality  urban_population_growth  tourists  
0              25.7                

In [9]:
na_mask = cities_v5["gdp"].isna()
na_countries = (
    cities_v5[na_mask][["country", "country_for_macro", "country_for_macro_norm"]]
    .drop_duplicates()
    .sort_values("country")
)

print("\nCountries with NaN macro after mapping (should be mostly US state codes):")
print(na_countries.to_string(index=False))


Countries with NaN macro after mapping (should be mostly US state codes):
          country country_for_macro country_for_macro_norm
               AB                AB                     ab
               AK                AK                     ak
               AZ                AZ                     az
               BC                BC                     bc
               CA                CA                     ca
               CO                CO                     co
               DC                DC                     dc
               FL                FL                     fl
               GA                GA                     ga
               HI                HI                     hi
Hong Kong (China) Hong Kong (China)      hong kong (china)
               ID                ID                     id
               IL                IL                     il
               IN                IN                     in
               KY                KY     

In [ ]:
# Save final v5 training table (v4 + macro features)
out_path_v5 = "../data/compiled_model_ready/MR_cities_worldpop_knn_macro_v5.csv"
cities_v5.to_csv(out_path_v5, index=False)
print("Saved merged v5 city table to:", out_path_v5)